# 🖼️ Notebook 6 — Milvus Walkthrough (Read-Only)

> **No install required.** This notebook is a guided tour of how Milvus is used at scale for **image search**. Read top-to-bottom; every code block is illustrative.

## 🎯 What you'll learn
- Why Milvus is the choice for **billion-scale** workloads
- How the architecture splits responsibilities across components
- The full code path for a CLIP-based **image search** application
- Index-type tradeoffs: `IVF_FLAT`, `IVF_PQ`, `HNSW`, `DiskANN`

## 🍱 Analogy
**Milvus = Kafka for vectors.**
- Multiple specialized nodes (proxy, coordinator, query, data, index).
- Built for distributed deployment from day one.
- Overkill for laptop demos, perfect for production at scale.

In [ ]:
from IPython.display import SVG
SVG("""<svg viewBox="0 0 720 230" xmlns="http://www.w3.org/2000/svg" width="720"><style>text{font-family:Inter,system-ui,sans-serif}</style><rect width="720" height="230" fill="#f8fafc" rx="12"/><rect x="20" y="60" width="140" height="80" rx="10" fill="#dbeafe" stroke="#2563eb"/><text x="90" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#1e3a8a">Image bytes</text><text x="90" y="110" text-anchor="middle" font-size="11" fill="#1e3a8a">millions of files</text><text x="90" y="125" text-anchor="middle" font-size="11" fill="#1e3a8a">S3 / blob</text><rect x="200" y="60" width="140" height="80" rx="10" fill="#ede9fe" stroke="#7c3aed"/><text x="270" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#5b21b6">CLIP encoder</text><text x="270" y="110" text-anchor="middle" font-size="11" fill="#5b21b6">image → 512-d</text><text x="270" y="125" text-anchor="middle" font-size="11" fill="#5b21b6">GPU batch</text><rect x="380" y="60" width="140" height="80" rx="10" fill="#fef3c7" stroke="#f59e0b"/><text x="450" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#78350f">Milvus cluster</text><text x="450" y="110" text-anchor="middle" font-size="11" fill="#78350f">shards + replicas</text><text x="450" y="125" text-anchor="middle" font-size="11" fill="#78350f">GPU IVF / DiskANN</text><rect x="560" y="60" width="140" height="80" rx="10" fill="#dcfce7" stroke="#15803d"/><text x="630" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#065f46">Search service</text><text x="630" y="110" text-anchor="middle" font-size="11" fill="#065f46">ANN + filters</text><text x="630" y="125" text-anchor="middle" font-size="11" fill="#065f46">p99 latency SLA</text><text x="172" y="105" font-size="22" fill="#475569">→</text><text x="352" y="105" font-size="22" fill="#475569">→</text><text x="532" y="105" font-size="22" fill="#475569">→</text><text x="360" y="35" text-anchor="middle" font-size="14" font-weight="700" fill="#0f172a">Milvus architecture</text><text x="360" y="180" text-anchor="middle" font-size="11" fill="#475569">Milvus is built for billion-scale, GPU-accelerated, distributed vector search</text></svg>""")

## 🧱 Architecture in one paragraph
A **proxy** receives requests. A **coordinator** plans where data goes. **Data nodes** persist vectors to object storage (S3, MinIO). **Index nodes** build indexes asynchronously. **Query nodes** load index segments into memory and serve searches. This separation lets you scale each layer independently — e.g., add more query nodes during traffic spikes without redeploying anything else.

## 1️⃣ Connect & create a collection (illustrative)

```python
from pymilvus import MilvusClient, DataType

client = MilvusClient(uri="http://milvus-prod.internal:19530", token="user:pass")

schema = client.create_schema(auto_id=True, enable_dynamic_field=True)
schema.add_field("pk",        DataType.INT64,         is_primary=True)
schema.add_field("embedding", DataType.FLOAT_VECTOR,  dim=512)        # CLIP ViT-B/32
schema.add_field("path",      DataType.VARCHAR,       max_length=512)
schema.add_field("label",     DataType.VARCHAR,       max_length=64)
schema.add_field("uploaded",  DataType.INT64)                          # epoch seconds

client.create_collection(collection_name="image_search", schema=schema)
```

## 2️⃣ Encode images with CLIP (illustrative)

```python
import torch, clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

def encode_image_batch(paths):
    images = torch.stack([preprocess(Image.open(p)) for p in paths]).to(device)
    with torch.no_grad():
        feats = model.encode_image(images)
        feats /= feats.norm(dim=-1, keepdim=True)   # L2-normalize for cosine
    return feats.cpu().numpy()
```

In production you'd run this across a Spark/Ray cluster with hundreds of GPUs to embed millions of images per hour.

## 3️⃣ Bulk insert (illustrative)

```python
batch = []
for path, label in image_iterator:                    # millions of items
    batch.append({"path": path, "label": label, "uploaded": int(time.time())})
    if len(batch) == 1000:
        vecs = encode_image_batch([b["path"] for b in batch])
        for b, v in zip(batch, vecs):
            b["embedding"] = v.tolist()
        client.insert(collection_name="image_search", data=batch)
        batch.clear()
```

Milvus accepts ~10k vectors/sec/node and handles back-pressure transparently.

## 4️⃣ Build an index — pick the right type

| Index | Memory | Build time | Recall | When |
|-------|--------|------------|--------|------|
| `IVF_FLAT`   | High | Fast    | High   | <10M, RAM available |
| `IVF_PQ`     | Low  | Fast    | Medium | 10M–1B, save RAM |
| `HNSW`       | High | Slow    | Highest| Latency-critical |
| `DiskANN`    | SSD  | Slow    | High   | Billions on disk |

```python
client.create_index(
    collection_name="image_search",
    field_name="embedding",
    index_params={
        "metric_type": "COSINE",
        "index_type":  "IVF_PQ",
        "params": {"nlist": 4096, "m": 16, "nbits": 8},
    },
)
client.load_collection("image_search")    # pull index into query-node memory
```

## 5️⃣ Search with filters (illustrative)

```python
query_vec = encode_image_batch(["./query.jpg"])[0].tolist()

results = client.search(
    collection_name="image_search",
    data=[query_vec],
    limit=20,
    search_params={"metric_type": "COSINE", "params": {"nprobe": 32}},
    filter='label == "shoes" and uploaded > 1700000000',
    output_fields=["path", "label", "uploaded"],
)
for hit in results[0]:
    print(hit["path"], "score=", hit["distance"])
```

`nprobe` controls how many IVF clusters are searched — higher = more recall, more latency.

## 6️⃣ Operational concerns

- **Sharding** — collections are split across shards by primary key hash.
- **Replication** — read replicas serve queries during peak traffic.
- **Consistency** — choose `Strong`, `Session`, or `Eventually` per query.
- **Compaction** — small segments are periodically merged; happens automatically.
- **Monitoring** — Prometheus metrics for QPS, latency p50/p99, segment count.

## 🎓 Recap — When to use Milvus
✅ Billion-scale workloads, GPU-accelerated indexing
✅ Need true horizontal scaling and multi-region replication
✅ Image / video / audio search at production scale
❌ Laptop prototypes → use Chroma or FAISS
❌ Want a fully-managed cloud service → use Pinecone or Zilliz Cloud (managed Milvus)

## 📚 Further reading
- https://milvus.io/docs
- https://github.com/milvus-io/bootcamp